# Experiment 8: Scraped multi-domain technical-documentation benchmark (Groq, free tier, fully resumable)

Scales the original setup (1 domain, 60 docs, 180 questions) to **8 documentation domains** in one shared corpus, including
same-category hard negatives (Nginx vs Apache httpd, Docker vs Kubernetes, PostgreSQL vs Python stdlib ...).

Pipeline: polite crawl (robots.txt respected) -> chunk -> near-duplicate removal -> **query generation with Groq (free)** in three styles
(keyword / troubleshooting / semantic) -> retrieval on the union corpus -> per-domain + leave-one-domain-out router analysis.

**Everything is checkpointed to Google Drive** (`AI_Project/RAG Research/Exp8`): crawl cache, chunks, generated questions (appended one by
one), per-model retrieval results. If Colab disconnects or the Groq daily limit is hit, just reopen and **Run all again** - finished steps are skipped.

**Setup:** GPU runtime + Colab Secret `GROQ_API_KEY` (left sidebar > key icon > add secret, enable notebook access). Do NOT paste the key into a cell.

In [1]:
!pip install -q sentence-transformers scikit-learn scipy pandas matplotlib requests beautifulsoup4 groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.5 MB/s eta 0:00:00


## Shared helpers

In [2]:
import numpy as np, pandas as pd, re, json, time, os
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import KFold
from sklearn.ensemble import GradientBoostingRegressor

METHODS = ['dense', 'hybrid', 'rerank']          # ordered cheapest -> most expensive
LAMBDAS = np.r_[0.0, np.geomspace(0.005, 1.0, 14)]

# ---------------------------------------------------------------- BM25 (fast, sparse, vectorised)
class BM25Index:
    def __init__(self, docs, k1=1.2, b=0.75):
        self.cv = CountVectorizer(stop_words='english', lowercase=True)
        tf = self.cv.fit_transform(docs).tocsr().astype(np.float32)
        N = tf.shape[0]
        df = np.asarray((tf > 0).sum(axis=0)).ravel()
        self.idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
        dl = np.asarray(tf.sum(axis=1)).ravel()
        avgdl = max(dl.mean(), 1e-9)
        tf = tf.tocoo()
        denom = tf.data + k1 * (1 - b + b * dl[tf.row] / avgdl)
        data = tf.data * (k1 + 1) / denom * self.idf[tf.col]
        self.M = sparse.csr_matrix((data, (tf.row, tf.col)), shape=tf.shape)
        self.MT = self.M.T.tocsr()

    def score(self, queries, batch=256):
        out = []
        for s in range(0, len(queries), batch):
            Q = self.cv.transform(queries[s:s + batch]).astype(np.float32).tocsr()
            Q.data[:] = 1.0
            out.append((Q @ self.MT).toarray().astype(np.float32))
        return np.vstack(out)

    def query_stats(self, qs):
        an = self.cv.build_analyzer(); vocab = self.cv.vocabulary_
        rows = []
        for q in qs:
            toks = an(q)
            inv = [t for t in toks if t in vocab]
            idfs = [float(self.idf[vocab[t]]) for t in inv]
            rows.append(dict(
                n_tok=len(q.split()), n_terms=len(toks),
                frac_oov=1 - len(inv) / max(1, len(toks)),
                mean_idf=float(np.mean(idfs)) if idfs else 0.0,
                max_idf=max(idfs) if idfs else 0.0,
                has_digit=float(bool(re.search(r'\d', q))),
                has_special=float(bool(re.search(r'[_/\\.\-:=]', q))),
                n_chars=len(q)))
        return pd.DataFrame(rows)

# ---------------------------------------------------------------- array utils
def minmax_rows(S):
    mn = S.min(1, keepdims=True); mx = S.max(1, keepdims=True)
    return (S - mn) / (mx - mn + 1e-9)

def topk(S, k):
    k = min(k, S.shape[1])
    part = np.argpartition(-S, k - 1, axis=1)[:, :k]
    vals = np.take_along_axis(S, part, 1)
    order = np.argsort(-vals, axis=1)
    return np.take_along_axis(part, order, 1)

def topvals(S, k=20):
    k = min(k, S.shape[1])
    return -np.sort(-np.partition(S, S.shape[1] - k, axis=1)[:, -k:], axis=1)

# ---------------------------------------------------------------- metrics (graded relevance supported)
def ndcg_at_k(ranked, rel, k=10):
    dcg = sum(rel.get(i, 0) / np.log2(r + 2) for r, i in enumerate(ranked[:k]))
    ideal = sorted(rel.values(), reverse=True)[:k]
    idcg = sum(g / np.log2(r + 2) for r, g in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

def eval_lists(tops, rels, k=10):
    nd, rc, mr = [], [], []
    for lst, rel in zip(tops, rels):
        lst = list(lst[:k])
        nd.append(ndcg_at_k(lst, rel, k))
        pos = {i for i, g in rel.items() if g > 0}
        rc.append(len(pos & set(lst)) / max(1, len(pos)))
        r_ = 0.0
        for r, i in enumerate(lst):
            if i in pos:
                r_ = 1.0 / (r + 1); break
        mr.append(r_)
    return np.array(nd), np.array(rc), np.array(mr)

# ---------------------------------------------------------------- retrieval stack (Dense / Hybrid / Hybrid+Rerank)
def run_stack(model_name, doc_texts, q_texts, bm, reranker, exclude_idx=None, alpha=0.75,
              shortlist=20, q_prefix='', device=None, out_k=10):
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(model_name, device=device)
    model.max_seq_length = 256
    D = model.encode(doc_texts, batch_size=128, normalize_embeddings=True,
                     convert_to_numpy=True, show_progress_bar=True)
    Q = model.encode([q_prefix + q for q in q_texts], batch_size=128, normalize_embeddings=True,
                     convert_to_numpy=True, show_progress_bar=False)
    S_d = (Q @ D.T).astype(np.float32)
    S_b = bm.score(q_texts)
    if exclude_idx is not None:                       # ignore identical ids (e.g. ArguAna)
        for i, j in enumerate(exclude_idx):
            if j is not None:
                S_d[i, j] = S_d[i].min(); S_b[i, j] = S_b[i].min()
    H = alpha * minmax_rows(S_d) + (1 - alpha) * minmax_rows(S_b)
    top_d = topk(S_d, out_k)
    top_h_full = topk(H, shortlist)
    pairs = [(q_texts[i], doc_texts[j]) for i in range(len(q_texts)) for j in top_h_full[i]]
    sc = np.asarray(reranker.predict(pairs, batch_size=64, show_progress_bar=True)).reshape(len(q_texts), -1)
    order = np.argsort(-sc, axis=1)
    top_r = np.take_along_axis(top_h_full, order, 1)[:, :out_k]
    return dict(S_d=S_d, S_b=S_b, H=H, top_d=top_d, top_h=top_h_full[:, :out_k], top_r=top_r,
                model=model, D=D)

def measure_latency(model, D, bm, reranker, doc_texts, q_sample, alpha=0.75, shortlist=20, q_prefix=''):
    """Per-query wall-clock latency in ms for [dense, hybrid, rerank] (each includes the previous stage)."""
    for q in q_sample[:3]:
        model.encode([q_prefix + q], normalize_embeddings=True, show_progress_bar=False)
    td, th, tr = [], [], []
    for q in q_sample:
        t0 = time.perf_counter()
        qe = model.encode([q_prefix + q], normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
        sd = (qe @ D.T).astype(np.float32)
        _ = topk(sd, 10)
        t1 = time.perf_counter()
        sb = bm.score([q])
        h = alpha * minmax_rows(sd) + (1 - alpha) * minmax_rows(sb)
        sh = topk(h, shortlist)[0]
        t2 = time.perf_counter()
        _ = reranker.predict([(q, doc_texts[j]) for j in sh], show_progress_bar=False)
        t3 = time.perf_counter()
        td.append(t1 - t0); th.append(t2 - t0); tr.append(t3 - t0)
    return [1000 * float(np.mean(td)), 1000 * float(np.mean(th)), 1000 * float(np.mean(tr))]

# ---------------------------------------------------------------- label-free routing features
def build_features(S_d, S_b, H, q_texts, bm):
    d, b, h = topvals(S_d), topvals(S_b), topvals(H)
    kk = d.shape[1]
    f = pd.DataFrame({
        'd_top1': d[:, 0], 'd_margin': d[:, 0] - d[:, 1], 'd_spread': d[:, 0] - d[:, kk - 1], 'd_std': d.std(1),
        'b_top1': b[:, 0], 'b_margin': (b[:, 0] - b[:, 1]) / (b[:, 0] + 1e-6),
        'b_spread': (b[:, 0] - b[:, kk - 1]) / (b[:, 0] + 1e-6),
        'h_margin': h[:, 0] - h[:, 1], 'h_spread': h[:, 0] - h[:, kk - 1]})
    td, tb = topk(S_d, 10), topk(S_b, 10)
    f['overlap10'] = [len(set(a) & set(c)) / 10.0 for a, c in zip(td, tb)]
    f['top1_agree'] = (td[:, 0] == tb[:, 0]).astype(float)
    qs = bm.query_stats(q_texts)
    return pd.concat([qs.reset_index(drop=True), f.reset_index(drop=True)], axis=1)

# ---------------------------------------------------------------- routing analysis
def router_cost(choice, t):
    t = np.asarray(t, float)
    base = np.full(len(choice), t[1])                 # features need dense + BM25 scores
    base[choice == 2] = t[2]
    return float(base.mean())

def bootstrap_diff(a, b, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed); nq = len(a)
    idx = rng.integers(0, nq, size=(n_boot, nq))
    d = a[idx].mean(1) - b[idx].mean(1)
    lo, hi = np.percentile(d, [2.5, 97.5])
    return dict(diff=float(a.mean() - b.mean()), ci_lo=float(lo), ci_hi=float(hi), p_le0=float((d <= 0).mean()))

def fit_gain_models(Xtr, Ytr, seed=42):
    models = {}
    for m in (1, 2):
        reg = GradientBoostingRegressor(n_estimators=150, max_depth=2, learning_rate=0.05,
                                        subsample=0.8, random_state=seed)
        reg.fit(Xtr, Ytr[:, m] - Ytr[:, 0]); models[m] = reg
    return models

def predict_gains(models, Xte):
    P = np.zeros((len(Xte), 3))
    for m, reg in models.items():
        P[:, m] = reg.predict(Xte)
    return P

def choose(P, lam, t):
    cn = np.asarray(t, float) / t[2]
    U = P - lam * (cn - cn[0])[None, :]
    U[:, 0] = 0.0
    return U.argmax(1)

def _row(nd, rc, mr, cost):
    return dict(ndcg=float(nd.mean()), recall=float(rc.mean()), mrr=float(mr.mean()), cost=float(cost))

def analyze_routing(ND, RC, MR, t, X, seed=42, n_boot=5000, n_splits=5):
    nq = len(ND); ar = np.arange(nq); t = np.asarray(t, float)
    Xv = X.values.astype(float)
    res = {'n_queries': nq, 't': t.tolist()}
    best_fixed = int(ND.mean(0).argmax())
    tab = {}
    for i, name in enumerate(['Dense', 'Hybrid', 'Rerank']):
        tab[name] = _row(ND[:, i], RC[:, i], MR[:, i], t[i])
    # rule router (label-free, query text only)
    ntok = X['n_tok'].values; has_d = X['has_digit'].values > 0
    rc_ = np.where(has_d & (ntok < 12), 0, np.where(ntok >= 14, 1, 2))
    tab['Rule router'] = _row(ND[ar, rc_], RC[ar, rc_], MR[ar, rc_], t[rc_].mean())
    # oracle (ties -> cheapest because columns are ordered by cost)
    oc = ND.argmax(1)
    tab['Oracle'] = _row(ND[ar, oc], RC[ar, oc], MR[ar, oc], t[oc].mean())
    res['oracle_label_counts'] = {METHODS[i]: int((oc == i).sum()) for i in range(3)}
    res['n_informative'] = int((ND.max(1) - ND.min(1) > 1e-9).sum())
    # learned router: out-of-fold predicted gain over dense
    P = np.zeros((nq, 3))
    kf = KFold(n_splits=min(n_splits, nq), shuffle=True, random_state=seed)
    for tr, te in kf.split(Xv):
        P[te] = predict_gains(fit_gain_models(Xv[tr], ND[tr], seed), Xv[te])
    sweep = []
    for lam in LAMBDAS:
        ch = choose(P, lam, t)
        r = _row(ND[ar, ch], RC[ar, ch], MR[ar, ch], router_cost(ch, t))
        r.update(lam=float(lam), frac_rerank=float((ch == 2).mean()), frac_dense=float((ch == 0).mean()))
        sweep.append(r)
    ch0 = choose(P, 0.0, t)
    tab['Learned router'] = _row(ND[ar, ch0], RC[ar, ch0], MR[ar, ch0], router_cost(ch0, t))
    res['learned_sweep'] = sweep
    res['best_fixed'] = ['Dense', 'Hybrid', 'Rerank'][best_fixed]
    res['boot_learned_vs_best_fixed'] = bootstrap_diff(ND[ar, ch0], ND[:, best_fixed], n_boot, seed)
    bf_nd = ND[:, best_fixed].mean()
    ok = [s['cost'] for s in sweep if s['ndcg'] >= bf_nd]
    res['min_cost_to_match_best_fixed'] = float(min(ok)) if ok else None
    res['oracle_gap_closed'] = float((tab['Learned router']['ndcg'] - bf_nd) /
                                     max(tab['Oracle']['ndcg'] - bf_nd, 1e-9))
    # confidence-threshold (score-margin) baselines, swept as Pareto curves
    conf = X['d_margin'].values; order = np.argsort(conf)
    curves = {}
    for name, base in [('Margin router (dense/rerank)', 0), ('Margin router (hybrid/rerank)', 1)]:
        pts = []
        for q in np.linspace(0, 1, 21):
            k = int(round(q * nq)); ch = np.full(nq, base); ch[order[:k]] = 2
            pts.append(dict(q=float(q), ndcg=float(ND[ar, ch].mean()), cost=float(t[ch].mean())))
        curves[name] = pts
    res['margin_curves'] = curves
    res['table'] = tab
    res['_P'] = P
    return res

def lodo_routing(sets, n_boot=5000, seed=42):
    """Leave-one-set-out: train the gain regressors on all other sets, test on the held-out one."""
    out = {}
    names = list(sets.keys())
    if len(names) < 2:
        return out
    for held in names:
        tr_names = [k for k in names if k != held]
        Xtr = np.vstack([sets[k]['X'].values.astype(float) for k in tr_names])
        Ytr = np.vstack([sets[k]['ND'] for k in tr_names])
        models = fit_gain_models(Xtr, Ytr, seed)
        S = sets[held]; ND = S['ND']; ar = np.arange(len(ND)); t = np.asarray(S['t'], float)
        P = predict_gains(models, S['X'].values.astype(float))
        sweep = []
        for lam in LAMBDAS:
            ch = choose(P, lam, t)
            sweep.append(dict(lam=float(lam), ndcg=float(ND[ar, ch].mean()), cost=router_cost(ch, t)))
        bf = int(ND.mean(0).argmax()); ch0 = choose(P, 0.0, t)
        out[held] = dict(best_fixed=['Dense', 'Hybrid', 'Rerank'][bf], best_fixed_ndcg=float(ND[:, bf].mean()),
                         router_ndcg=float(ND[ar, ch0].mean()), sweep=sweep,
                         boot=bootstrap_diff(ND[ar, ch0], ND[:, bf], n_boot, seed))
    return out

# ---------------------------------------------------------------- reporting (tables, LaTeX, figures)
ROW_ORDER = ['Dense', 'Hybrid', 'Rerank', 'Rule router', 'Learned router', 'Oracle']

def main_table(RES_model, metric='ndcg'):
    dsets = list(RES_model.keys())
    df = pd.DataFrame({d: {r: RES_model[d]['table'][r][metric] for r in ROW_ORDER} for d in dsets})
    df['Avg'] = df.mean(axis=1)
    if metric != 'cost':
        df['Latency (ms)'] = pd.Series({r: np.mean([RES_model[d]['table'][r]['cost'] for d in dsets]) for r in ROW_ORDER})
    return df.loc[ROW_ORDER]

def _esc(s):
    return str(s).replace('_', '\\_').replace('%', '\\%').replace('&', '\\&')

def df_to_latex(df, caption, label, fmt='{:.3f}', lat_fmt='{:.1f}'):
    cols = list(df.columns)
    best = {c: df.loc[[r for r in df.index if r != 'Oracle'], c].idxmax() for c in cols if c != 'Latency (ms)'}
    lines = ['\\begin{table}[t]', '\\centering', '\\caption{' + _esc(caption) + '}', '\\label{' + label + '}',
             '\\begin{tabular}{l' + 'c' * len(cols) + '}', '\\toprule',
             ' & ' + ' & '.join(_esc(c) for c in cols) + ' \\\\', '\\midrule']
    for r in df.index:
        cells = []
        for c in cols:
            v = df.loc[r, c]
            s = lat_fmt.format(v) if c == 'Latency (ms)' else fmt.format(v)
            if c != 'Latency (ms)' and best.get(c) == r:
                s = '\\textbf{' + s + '}'
            cells.append(s)
        lines.append(_esc(r) + ' & ' + ' & '.join(cells) + ' \\\\')
        if r == 'Rerank':
            lines.append('\\midrule')
    lines += ['\\bottomrule', '\\end{tabular}', '\\end{table}']
    return '\n'.join(lines)

def significance_table(RES):
    rows = []
    for m, dd in RES.items():
        for d, r in dd.items():
            b = r['boot_learned_vs_best_fixed']
            rows.append(dict(model=m, dataset=d, best_fixed=r['best_fixed'],
                             best_fixed_ndcg=r['table'][r['best_fixed']]['ndcg'],
                             learned_ndcg=r['table']['Learned router']['ndcg'], diff=b['diff'],
                             ci_lo=b['ci_lo'], ci_hi=b['ci_hi'], significant=bool(b['ci_lo'] > 0),
                             oracle_ndcg=r['table']['Oracle']['ndcg'], oracle_gap_closed=r['oracle_gap_closed'],
                             min_cost_to_match=r['min_cost_to_match_best_fixed']))
    return pd.DataFrame(rows)

def lodo_table(LODO):
    rows = []
    for m, dd in LODO.items():
        for d, r in dd.items():
            rows.append(dict(model=m, held_out=d, best_fixed=r['best_fixed'], best_fixed_ndcg=r['best_fixed_ndcg'],
                             router_ndcg=r['router_ndcg'], diff=r['boot']['diff'], ci_lo=r['boot']['ci_lo'],
                             ci_hi=r['boot']['ci_hi'], significant=bool(r['boot']['ci_lo'] > 0)))
    return pd.DataFrame(rows)

def plot_pareto(RES, path_prefix):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    models = list(RES.keys()); dsets = list(RES[models[0]].keys())
    fig, axes = plt.subplots(len(models), len(dsets), figsize=(3.3 * len(dsets), 2.7 * len(models)), squeeze=False)
    for i, m in enumerate(models):
        for j, d in enumerate(dsets):
            ax = axes[i][j]; r = RES[m][d]
            sw = r['learned_sweep']
            ax.plot([s['cost'] for s in sw], [s['ndcg'] for s in sw], '-o', ms=3, lw=1.2, label='Learned router')
            for name, pts in r['margin_curves'].items():
                ax.plot([p['cost'] for p in pts], [p['ndcg'] for p in pts], '--', lw=1, label=name)
            for name, mk in [('Dense', 's'), ('Hybrid', '^'), ('Rerank', 'D')]:
                ax.scatter(r['table'][name]['cost'], r['table'][name]['ndcg'], marker=mk, s=28, zorder=5, label=name)
            ax.scatter(r['table']['Oracle']['cost'], r['table']['Oracle']['ndcg'], marker='*', s=70, c='k', zorder=6, label='Oracle')
            ax.set_xscale('log')
            if i == 0: ax.set_title(d, fontsize=9)
            if j == 0: ax.set_ylabel(m + '\nnDCG@10', fontsize=8)
            if i == len(models) - 1: ax.set_xlabel('Latency (ms/query)', fontsize=8)
            ax.tick_params(labelsize=7); ax.grid(alpha=0.3)
    h, l = axes[0][0].get_legend_handles_labels()
    fig.legend(h, l, loc='lower center', ncol=4, fontsize=7, frameon=False)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    fig.savefig(path_prefix + '.pdf'); fig.savefig(path_prefix + '.png', dpi=300)
    plt.close(fig)

def to_jsonable(o):
    if isinstance(o, dict):
        return {str(k): to_jsonable(v) for k, v in o.items() if not str(k).startswith('_')}
    if isinstance(o, (list, tuple)):
        return [to_jsonable(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    return o


## Config (edit domains freely)

In [3]:
import torch, random, gc, pickle, requests, urllib.robotparser
from urllib.parse import urljoin, urldefrag
from bs4 import BeautifulSoup
from sentence_transformers import CrossEncoder
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/AI_Project/RAG Research/Exp8'
os.makedirs(OUT, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DOMAINS = {   # name: (seed url, url prefix that the crawl must stay inside)
    'nginx':      ('https://nginx.org/en/docs/', 'https://nginx.org/en/docs/'),
    'apache':     ('https://httpd.apache.org/docs/2.4/', 'https://httpd.apache.org/docs/2.4/'),
    'postgresql': ('https://www.postgresql.org/docs/current/index.html', 'https://www.postgresql.org/docs/current/'),
    'docker':     ('https://docs.docker.com/engine/', 'https://docs.docker.com/engine/'),
    'kubernetes': ('https://kubernetes.io/docs/concepts/', 'https://kubernetes.io/docs/concepts/'),
    'git':        ('https://git-scm.com/docs', 'https://git-scm.com/docs'),
    'python':     ('https://docs.python.org/3/library/index.html', 'https://docs.python.org/3/library/'),
    'prometheus': ('https://prometheus.io/docs/introduction/overview/', 'https://prometheus.io/docs/'),
}
MAX_PAGES = 120           # per domain
CRAWL_DELAY = 0.5
CHUNK_WORDS = 180
N_Q_PER_DOMAIN = 100      # 800 questions total; fits Groq free daily token budget. Raise later and re-run to add more.
STYLES = ['keyword', 'troubleshooting', 'semantic']
GEN_MODEL = 'llama-3.1-8b-instant'   # Groq free tier; if Groq retires it, pick another chat model from console.groq.com/docs/models
TPM_BUDGET = 5000                    # tokens/min we allow ourselves (free tier is ~6000 TPM for this model; check console.groq.com/settings/limits)
ALLOW_PARTIAL = False                # True = continue to experiments even if not all questions are generated yet
MODELS = {'MiniLM-L6': 'sentence-transformers/all-MiniLM-L6-v2',
          'mpnet-base': 'sentence-transformers/all-mpnet-base-v2',
          'bge-small': 'BAAI/bge-small-en-v1.5'}
QUERY_PREFIX = {'bge-small': 'Represent this sentence for searching relevant passages: '}
ALPHA, SHORTLIST, N_LAT = 0.75, 20, 50
HEADERS = {'User-Agent': 'academic-rag-benchmark-crawler (research use; contact: shashikantkaushik4@gmail.com)'}
CONFIG = dict(domains={k: v[0] for k, v in DOMAINS.items()}, max_pages=MAX_PAGES, chunk_words=CHUNK_WORDS,
              n_q_per_domain=N_Q_PER_DOMAIN, gen_model=GEN_MODEL, gen_provider='groq', models=MODELS, alpha=ALPHA,
              shortlist=SHORTLIST, device=DEVICE, gpu=(torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu'))
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=256, device=DEVICE)

Mounted at /content/drive


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## Step 1: Crawl (robots.txt respected; per-domain cache on Drive, so it resumes after a disconnect)

In [4]:
SKIP_EXT = ('.pdf', '.zip', '.gz', '.png', '.jpg', '.jpeg', '.gif', '.svg', '.tar', '.xz', '.css', '.js')

def get_robots(base):
    rp = urllib.robotparser.RobotFileParser()
    try:
        rp.set_url(urljoin(base, '/robots.txt')); rp.read(); return rp
    except Exception:
        return None

def extract(soup):
    title = soup.title.get_text(' ', strip=True) if soup.title else ''
    for tag in soup(['script', 'style', 'nav', 'header', 'footer', 'aside', 'form', 'noscript']):
        tag.decompose()
    main = soup.find('main') or soup.find('article') or soup.find('div', id='content') or soup.body
    text = main.get_text('\n') if main else ''
    lines = [re.sub(r'\s+', ' ', l).strip() for l in text.split('\n')]
    return title, [l for l in lines if len(l.split()) >= 2]

def crawl(seed, prefix, max_pages, delay):
    rp = get_robots(seed); seen, queue, pages = set(), [seed], []
    while queue and len(pages) < max_pages:
        url = urldefrag(queue.pop(0))[0]
        if url in seen or not url.startswith(prefix) or url.lower().endswith(SKIP_EXT):
            continue
        seen.add(url)
        if rp is not None and not rp.can_fetch(HEADERS['User-Agent'], url):
            continue
        try:
            r = requests.get(url, headers=HEADERS, timeout=20)
        except Exception:
            continue
        if r.status_code != 200 or 'text/html' not in r.headers.get('Content-Type', ''):
            continue
        soup = BeautifulSoup(r.text, 'html.parser')
        for a in soup.find_all('a', href=True):
            u = urldefrag(urljoin(url, a['href']))[0]
            if u.startswith(prefix) and u not in seen:
                queue.append(u)
        title, lines = extract(soup)
        pages.append(dict(url=url, title=title, lines=lines))
        time.sleep(delay)
    return pages

CACHE = f'{OUT}/crawl_cache.json'
crawled = json.load(open(CACHE)) if os.path.exists(CACHE) else {}
for name, (seed, prefix) in DOMAINS.items():
    if name in crawled:
        print(name, 'cached:', len(crawled[name]), 'pages'); continue
    crawled[name] = crawl(seed, prefix, MAX_PAGES, CRAWL_DELAY)
    json.dump(crawled, open(CACHE, 'w'))          # saved after EVERY domain
    print(name, ':', len(crawled[name]), 'pages (saved to Drive)')

nginx cached: 120 pages
apache cached: 120 pages
postgresql cached: 120 pages
docker cached: 120 pages
kubernetes cached: 120 pages
git cached: 0 pages
python cached: 120 pages
prometheus cached: 0 pages


## Step 2: Chunk + de-duplicate (saved to Drive so chunk ids stay stable across restarts)

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

def chunk_lines(lines, target=CHUNK_WORDS, min_words=60, max_words=400):
    chunks, buf, n = [], [], 0
    for l in lines:
        buf.append(l); n += len(l.split())
        if n >= target:
            chunks.append(' '.join(' '.join(buf).split()[:max_words])); buf, n = [], 0
    if n >= min_words:
        chunks.append(' '.join(' '.join(buf).split()[:max_words]))
    return chunks

def dedup(texts, thr=0.9):
    X = TfidfVectorizer(stop_words='english', sublinear_tf=True).fit_transform(texts)
    keep = np.ones(len(texts), bool)
    for s in range(0, len(texts), 500):
        blk = (X[s:s + 500] @ X.T).toarray()
        for r in range(blk.shape[0]):
            i = s + r
            if not keep[i]: continue
            dup = np.where(blk[r] > thr)[0]; dup = dup[dup > i]
            keep[dup] = False
    return keep

CHUNK_FILE = f'{OUT}/chunks.jsonl'
if os.path.exists(CHUNK_FILE):
    chunks = [json.loads(l) for l in open(CHUNK_FILE, encoding='utf-8')]
    print('loaded', len(chunks), 'chunks from Drive')
else:
    chunks = []
    for name, pages in crawled.items():
        for p in pages:
            for c in chunk_lines(p['lines']):
                chunks.append(dict(domain=name, url=p['url'], title=p['title'], text=c))
    keep = dedup([c['text'] for c in chunks])
    chunks = [c for c, k in zip(chunks, keep) if k]
    for i, c in enumerate(chunks):
        c['id'] = i
        c['doc'] = (c['title'] + '. ' + c['text']).strip()
    with open(CHUNK_FILE, 'w', encoding='utf-8') as f:
        for c in chunks: f.write(json.dumps(c) + '\n')
    print(len(chunks), 'chunks after dedup (saved)')
print(pd.Series([c['domain'] for c in chunks]).value_counts().to_string())

loaded 5585 chunks from Drive
python        1829
kubernetes     995
apache         907
docker         743
nginx          673
postgresql     438


## Step 3: Generate queries with Groq (free) - resumable, batched
Each accepted question is appended to `qa_generated.jsonl` on Drive immediately. Re-running this cell continues where it stopped - no progress is lost on a disconnect.

Generates up to 3 questions per Groq call (instead of 1) to cut wall-clock time roughly 3x. If the model's JSON formatting doesn't parse after a few tries, that batch automatically falls back to one-call-per-question for just those jobs, so robustness is unchanged - only the common case gets faster.

Throttled to stay under the free-tier tokens/minute limit; if the **daily** limit is hit the cell saves and stops - run it again after the limit resets.


In [6]:
import groq
from groq import Groq
try:
    from google.colab import userdata
    API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    import getpass; API_KEY = getpass.getpass('Groq API key: ')
client = Groq(api_key=API_KEY)

STYLE_PROMPTS = {
 'keyword': "Write ONE short search query (2-8 words) that a sysadmin would type into a search box to find this passage. It must contain at least one exact technical token from the passage (a directive, flag, function name, error code or config key). No question words.",
 'troubleshooting': "Write ONE long, realistic query (15-35 words) in which a user describes a concrete problem, symptom or error they are seeing, whose answer is contained in this passage. Do not name the answer and do not copy sentences from the passage.",
 'semantic': "Write ONE natural-language question (8-20 words) that this passage answers, phrased conceptually. Avoid reusing the passage's distinctive technical terms; paraphrase them.",
}
WLIM = {'keyword': (2, 10), 'troubleshooting': (12, 45), 'semantic': (6, 26)}
tok = lambda s: set(re.findall(r'[a-z0-9_]+', s.lower()))
SKIP = object()
BATCH_SIZE = 3   # questions generated per Groq call - cuts wall-clock time roughly 3x vs one-per-call

class DailyLimit(Exception): pass

_win = []   # (timestamp, tokens) used in the last 60 s
def wait_for_tpm(est=550):
    while True:
        now = time.time(); _win[:] = [(t, n) for t, n in _win if now - t < 60]
        if sum(n for _, n in _win) + est <= TPM_BUDGET: return
        time.sleep(1.0)

def clean_q(q):
    q = q.strip().split('\n')[0].strip().strip('"').strip("'").strip()
    return re.sub(r'^(here.{0,40}?:|query:|question:|search query:)\s*', '', q, flags=re.I).strip().strip('"')

def valid(q, style, c):
    lo, hi = WLIM[style]
    if not (lo <= len(q.split()) <= hi): return False
    if style == 'semantic' and len(tok(q) & tok(c['text'])) / max(1, len(tok(q))) > 0.6: return False
    return True

def _call_groq(prompt, max_tokens):
    """One Groq call with rate-limit / transient-error handling. Raises DailyLimit on a long retry-after."""
    for attempt in range(6):
        try:
            r = client.chat.completions.create(model=GEN_MODEL, messages=[{'role': 'user', 'content': prompt}],
                                               max_tokens=max_tokens, temperature=0.7)
            return r
        except groq.RateLimitError as e:
            ra = float(e.response.headers.get('retry-after', 30) or 30)
            if ra > 300: raise DailyLimit(f'retry-after {ra:.0f}s')
            time.sleep(ra + 1); continue
        except (groq.APIConnectionError, groq.APITimeoutError, groq.InternalServerError):
            time.sleep(5); continue
    return None

def gen_single(c, style):
    """Original one-question-per-call path. Used as the batch cell's fallback when batched
    JSON parsing repeatedly fails, so a formatting hiccup never permanently stalls a job."""
    prompt = (f"{STYLE_PROMPTS[style]}\n\nPassage (from the page titled '{c['title']}'):\n\"\"\"\n{c['text'][:1200]}\n\"\"\"\n\n"
              "Output ONLY the query text on one line - no quotes, no label, no explanation.")
    invalid = 0
    for attempt in range(6):
        wait_for_tpm()
        r = _call_groq(prompt, 80)
        if r is None: continue
        _win.append((time.time(), r.usage.total_tokens))
        q = clean_q(r.choices[0].message.content or '')
        if valid(q, style, c): return q
        invalid += 1
        if invalid >= 3: return None
    return SKIP

def _extract_json_array(raw, expected_len):
    raw = raw.strip()
    raw = re.sub(r'^```(?:json)?\s*', '', raw)
    raw = re.sub(r'\s*```$', '', raw)
    for candidate in (raw, raw[raw.find('['):raw.rfind(']')+1] if '[' in raw and ']' in raw else ''):
        if not candidate: continue
        try:
            arr = json.loads(candidate)
            if isinstance(arr, list) and len(arr) == expected_len:
                return arr
        except Exception:
            pass
    return None

def gen_batch(jobs_batch, chunks):
    """Generate up to BATCH_SIZE questions in ONE Groq call (one network round-trip instead of
    len(jobs_batch)). Falls back to gen_single per-job if the model's JSON formatting doesn't
    parse after a few tries, so robustness is unchanged - only the common case gets faster."""
    parts = []
    for i, (cid, st) in enumerate(jobs_batch, 1):
        c = chunks[cid]
        parts.append(f"Task {i} (style: {st}):\n{STYLE_PROMPTS[st]}\nPassage {i} (from '{c['title']}'):\n\"\"\"\n{c['text'][:900]}\n\"\"\"\n")
    prompt = ("You will generate one query for each numbered task below, following that task's instructions exactly.\n\n"
              + "\n\n".join(parts) +
              f"\nOutput ONLY a JSON array of {len(jobs_batch)} strings, one per task in order, nothing else."
              f" Example: [\"query1\", \"query2\"]")

    for attempt in range(4):
        wait_for_tpm(est=220*len(jobs_batch) + 250)
        r = _call_groq(prompt, 90*len(jobs_batch) + 60)
        if r is None: continue
        _win.append((time.time(), r.usage.total_tokens))
        arr = _extract_json_array(r.choices[0].message.content or '', len(jobs_batch))
        if arr is None:
            continue   # malformed JSON this attempt - retry the whole batch
        out = []
        for (cid, st), q_raw in zip(jobs_batch, arr):
            q = clean_q(str(q_raw))
            out.append(q if valid(q, st, chunks[cid]) else None)
        return out

    # Batch JSON formatting never worked after 4 tries -> fall back to one-per-call for this batch,
    # so these jobs are not permanently stuck just because the model misformatted the array once.
    return [gen_single(chunks[cid], st) for cid, st in jobs_batch]

JOBS_FILE = f'{OUT}/qa_jobs.json'
if os.path.exists(JOBS_FILE):
    jobs = json.load(open(JOBS_FILE))
else:
    rng = random.Random(42); jobs = []
    for name in DOMAINS:
        ids_ = [c['id'] for c in chunks if c['domain'] == name]
        pick = rng.sample(ids_, min(N_Q_PER_DOMAIN, len(ids_)))
        jobs += [[cid, STYLES[k % 3]] for k, cid in enumerate(pick)]
    json.dump(jobs, open(JOBS_FILE, 'w'))

GEN_FILE = f'{OUT}/qa_generated.jsonl'
done = {}
if os.path.exists(GEN_FILE):
    for l in open(GEN_FILE, encoding='utf-8'):
        x = json.loads(l); done[(x['expected_chunk_id'], x['category'])] = x
todo = [j for j in jobs if (j[0], j[1]) not in done]
print(f'{len(done)} already done, {len(todo)} remaining '
      f'(batched {BATCH_SIZE}/call; ~{5*BATCH_SIZE//3}-{6*BATCH_SIZE//3}s per call, ~{5/BATCH_SIZE:.1f}-{6/BATCH_SIZE:.1f}s/question amortized)')

t0 = time.time()
with open(GEN_FILE, 'a', encoding='utf-8') as f:
    i = 0
    while i < len(todo):
        batch = todo[i:i + BATCH_SIZE]
        try:
            results = gen_batch(batch, chunks)
        except DailyLimit as e:
            print('Groq daily limit reached (', e, '). Progress is saved. Re-run this cell after the limit resets.'); break
        for (cid, st), q in zip(batch, results):
            if q is SKIP: continue
            rec = dict(expected_chunk_id=cid, category=st, domain=chunks[cid]['domain'], question=q)
            f.write(json.dumps(rec) + '\n'); f.flush(); os.fsync(f.fileno())
            done[(cid, st)] = rec
        i += len(batch)
        if (i // BATCH_SIZE) % 10 == 0:
            print(f'  {i}/{len(todo)}  elapsed {(time.time() - t0) / 60:.1f} min')

job_keys = {(j[0], j[1]) for j in jobs}
QA = [v for kk, v in done.items() if kk in job_keys and v['question']]
remaining = len([j for j in jobs if (j[0], j[1]) not in done])
print(f'{len(QA)} valid questions | {remaining} jobs still not generated')
print(pd.crosstab(pd.Series([x['domain'] for x in QA]), pd.Series([x['category'] for x in QA])))
for x in random.Random(1).sample(QA, min(8, len(QA))):
    print(f"[{x['domain']}/{x['category']}] {x['question']}")
if remaining > 0 and not ALLOW_PARTIAL:
    raise RuntimeError('Question generation not finished yet - re-run this cell (later if the daily limit was hit), or set ALLOW_PARTIAL=True.')


Groq API key: ··········
150 already done, 450 remaining (batched 3/call; ~5-6s per call, ~1.7-2.0s/question amortized)


NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

## Step 4: Save the final dataset (release this + DOI)

In [ ]:
with open(f'{OUT}/corpus.jsonl', 'w', encoding='utf-8') as f:
    for c in chunks: f.write(json.dumps(dict(id=c['id'], domain=c['domain'], url=c['url'], title=c['title'], text=c['text'])) + '\n')
with open(f'{OUT}/qa.jsonl', 'w', encoding='utf-8') as f:
    for x in QA: f.write(json.dumps(x) + '\n')
print('saved corpus.jsonl and qa.jsonl to', OUT)
print('IMPORTANT: read ~50 random questions from qa.jsonl and note what fraction look wrong - report this audit rate in the paper.')

## Step 5: Retrieval experiments on the union corpus (checkpoint per model on Drive)

In [ ]:
docs = [c['doc'] for c in chunks]
qs = [x['question'] for x in QA]
rels = [{x['expected_chunk_id']: 1} for x in QA]
dom = np.array([x['domain'] for x in QA]); sty = np.array([x['category'] for x in QA])
bm = BM25Index(docs)

RES = {m: {} for m in MODELS}; SETS = {m: {} for m in MODELS}; STYLE_TAB = {}
for mname, mpath in MODELS.items():
    ck = f'{OUT}/stack_{mname}_{len(QA)}q.pkl'
    if os.path.exists(ck):
        ND, RC, MR, X, lat = pickle.load(open(ck, 'rb')); print('---', mname, ': loaded checkpoint from Drive')
    else:
        print('---', mname)
        pref = QUERY_PREFIX.get(mname, '')
        st = run_stack(mpath, docs, qs, bm, reranker, alpha=ALPHA, shortlist=SHORTLIST, q_prefix=pref, device=DEVICE)
        ev = [eval_lists(st[k], rels) for k in ('top_d', 'top_h', 'top_r')]
        ND = np.stack([e[0] for e in ev], 1); RC = np.stack([e[1] for e in ev], 1); MR = np.stack([e[2] for e in ev], 1)
        X = build_features(st['S_d'], st['S_b'], st['H'], qs, bm)
        lat = measure_latency(st['model'], st['D'], bm, reranker, docs, qs[:N_LAT], alpha=ALPHA, shortlist=SHORTLIST, q_prefix=pref)
        pickle.dump((ND, RC, MR, X, lat), open(ck, 'wb'))
        del st; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
    for d in DOMAINS:
        mk = dom == d
        RES[mname][d] = analyze_routing(ND[mk], RC[mk], MR[mk], lat, X[mk].reset_index(drop=True))
        SETS[mname][d] = dict(ND=ND[mk], X=X[mk].reset_index(drop=True), t=lat)
    RES[mname]['ALL'] = analyze_routing(ND, RC, MR, lat, X)
    STYLE_TAB[mname] = pd.DataFrame({s: dict(zip(['dense', 'hybrid', 'rerank'], ND[sty == s].mean(0))) for s in STYLES}).T
    STYLE_TAB[mname]['oracle'] = [ND[sty == s].max(1).mean() for s in STYLES]
    print(STYLE_TAB[mname].round(3).to_string())
    print('latency ms [dense, hybrid, rerank]:', [round(x, 1) for x in lat])

## Step 6: Leave-one-domain-out router transfer + per-style tables

In [ ]:
LODO = {m: lodo_routing(SETS[m]) for m in MODELS}
print(lodo_table(LODO).round(3).to_string())
for m, t_ in STYLE_TAB.items():
    t_.to_csv(f'{OUT}/per_style_ndcg_{m}.csv')

## Tables, LaTeX, Pareto figure (all saved to Drive)

In [ ]:

os.makedirs(OUT, exist_ok=True)

for metric, mname_ in [('ndcg', 'nDCG@10'), ('recall', 'Recall@10'), ('mrr', 'MRR@10')]:
    for m in RES:
        df = main_table(RES[m], metric)
        df.to_csv(f'{OUT}/table_{metric}_{m}.csv')
        with open(f'{OUT}/table_{metric}_{m}.tex', 'w') as f:
            f.write(df_to_latex(df, f'{mname_} with {m} embeddings', f'tab:{metric}_{m}'))
        if metric == 'ndcg':
            print(f'\n===== {mname_} | {m} =====')
            print(df.round(3).to_string())

sig = significance_table(RES); sig.to_csv(f'{OUT}/significance_learned_vs_best_fixed.csv', index=False)
print('\n===== Learned router vs best fixed strategy (paired bootstrap, 95% CI) =====')
print(sig.round(3).to_string())

if LODO:
    lt = lodo_table(LODO); lt.to_csv(f'{OUT}/leave_one_out_router.csv', index=False)
    print('\n===== Leave-one-out router (trained on the other sets only) =====')
    print(lt.round(3).to_string())

plot_pareto(RES, f'{OUT}/pareto')
with open(f'{OUT}/all_results.json', 'w') as f:
    json.dump(to_jsonable({'results': RES, 'lodo': LODO, 'config': CONFIG}), f, indent=1)
print('\nAll outputs saved to Google Drive folder:', OUT)
print(sorted(os.listdir(OUT)))


## Write-up notes
- State clearly: queries are LLM-generated (Groq `llama-3.1-8b-instant`, prompts in this notebook), filtered by length / lexical-overlap rules, and manually audited (report the audit rate).
- Queries retrieve from the **union** corpus, so cross-domain confusion is part of the difficulty.
- The style label (keyword / troubleshooting / semantic) is created by the generator, so a router that exploits query style may look better here than on real users - say so; Exp7 (real queries) is the external-validity evidence.